In [1]:
%pip install --upgrade --quiet  xformers --quiet #TODO: try out xformer as this is supposed to be more memory efficient
%pip install --upgrade --quiet  langchain    --quiet
%pip install --upgrade --quiet  bitsandbytes --quiet
%pip install --upgrade --quiet  python-dotenv --quiet
%pip install accelerate --quiet



ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow-probability 0.22.0 requires typing-extensions<4.6.0, but you have typing-extensions 4.9.0 which is incompatible.
torchaudio 2.1.0+cu121 requires torch==2.1.0, but you have torch 2.2.0 which is incompatible.
torchdata 0.7.0 requires torch==2.1.0, but you have torch 2.2.0 which is incompatible.
torchtext 0.16.0 requires torch==2.1.0, but you have torch 2.2.0 which is incompatible.
torchvision 0.16.0+cu121 requires torch==2.1.0, but you have torch 2.2.0 which is incompatible.


In [ ]:
# %pip install --upgrade typing-extensions==4.5.0
# %pip install --upgrade torch==2.2.0
# %pip install --upgrade torchtext==0.16.0 torchvision==0.16.0+cu121 torchaudio==2.1.0+cu121
# %pip install --upgrade xformers langchain bitsandbytes python-dotenv accelerate

In [2]:
from langchain_community.llms.huggingface_pipeline import HuggingFacePipeline
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
import os
from dotenv import load_dotenv
import transformers
from torch import cuda, bfloat16


# load environment variables from .env file
from dotenv import load_dotenv, find_dotenv

_ = load_dotenv(find_dotenv())

bitsAndBites_config = transformers.BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=bfloat16,
)

# hf = HuggingFacePipeline.from_model_id(
#     model_id="cxllin/Llama2-7b-med-v1",
#     task="text-generation",
#     pipeline_kwargs={"max_new_tokens": 10
#                      trust_remote_code=True,
#                      quantization_config=bitsAndBites_config,
#                      device_map='auto',
#                      },
# )


# OR


model_id = "cxllin/Llama2-7b-med-v1"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    trust_remote_code=True,
    quantization_config=bitsAndBites_config,
    device_map="auto",
    token=os.environ.get("HF_AUTH"),
)

model.eval()

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:88: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


pytorch_model.bin.index.json:   0%|          | 0.00/26.8k [00:00<?, ?B/s]

pytorch_model-00001-of-00002.bin:   0%|          | 0.00/9.98G [00:00<?, ?B/s]

pytorch_model-00002-of-00002.bin:   0%|          | 0.00/3.50G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/174 [00:00<?, ?B/s]

/usr/local/lib/python3.10/dist-packages/transformers/generation/configuration_utils.py:392: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.9` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`. This was detected when initializing the generation config instance, which means the corresponding file may hold incorrect parameterization and should be fixed.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/transformers/generation/configuration_utils.py:397: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.6` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`. This was detected when initializing the generation config instance, which means the corresponding file may hold incorrect parameterization and should be fixed.
  warnings.warn(


In [3]:
model.get_memory_footprint()

3829940224

In [6]:
pipe = pipeline(
    task="text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=256,
    temperature=0.01,
    repetition_penalty=1.1,
)

hf = HuggingFacePipeline(pipeline=pipe)

In [7]:
from langchain.prompts import PromptTemplate

template = """Question: {question}

Answer: You are talking to a professional. Use elaborate langugage."""
prompt = PromptTemplate.from_template(template)

chain = prompt | hf

question = "What is electroencephalography?"

print(chain.invoke({"question": question}))

/usr/local/lib/python3.10/dist-packages/transformers/generation/configuration_utils.py:392: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.01` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/transformers/generation/configuration_utils.py:397: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.6` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(




Electroencephalography (EEG) is a non-invasive diagnostic tool that measures the electrical activity of the brain. It is used to diagnose and monitor various neurological conditions, including epilepsy, seizures, head injuries, and sleep disorders. EEG recordings are made by placing electrodes on the scalp or other parts of the body. The resulting data is then analyzed using specialized software to identify patterns and abnormalities in the brain's electrical activity. This information can be used to develop treatment plans for patients with these conditions.
